# 03 - Trade and Export Analysis

## Oman Economic Analysis Project

This notebook analyzes Oman's trade patterns, export/import trends, and trade balance.

### Key Questions
1. What are Oman's export and import trends?
2. How dependent is Oman on oil exports?
3. What is the current account balance trend?
4. How does Oman's trade compare with GCC peers?

In [ ]:
# Import libraries
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_processor import DataProcessor

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

In [ ]:
# Load data
processor = DataProcessor(data_dir='../data')
df = processor.load_data('gcc_economic_data.csv')
df = processor.clean_data(df)

print(f"Loaded {len(df):,} records")

## 1. Export and Import Trends

In [ ]:
# Export and Import indicators
export_ind = 'NE.EXP.GNFS.ZS'  # Exports as % of GDP
import_ind = 'NE.IMP.GNFS.ZS'  # Imports as % of GDP

oman_exports = df[(df['indicator_code'] == export_ind) & 
                  (df['country_code'] == 'OMN')].sort_values('year')
oman_imports = df[(df['indicator_code'] == import_ind) & 
                  (df['country_code'] == 'OMN')].sort_values('year')

fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(oman_exports['year'], oman_exports['value'], 
        marker='o', linewidth=2, label='Exports (% of GDP)', color='#2ca02c')
ax.plot(oman_imports['year'], oman_imports['value'], 
        marker='s', linewidth=2, label='Imports (% of GDP)', color='#d62728')

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('% of GDP', fontsize=12)
ax.set_title('Oman Exports vs Imports (% of GDP)', fontsize=14, fontweight='bold')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/charts/oman_trade_trends.png', dpi=150, bbox_inches='tight')
plt.show()

# Trade balance
print("\nTrade Balance (Exports - Imports as % of GDP):")
merged = pd.merge(oman_exports[['year', 'value']], 
                  oman_imports[['year', 'value']], 
                  on='year', suffixes=('_export', '_import'))
merged['trade_balance'] = merged['value_export'] - merged['value_import']
print(merged[['year', 'trade_balance']].tail(10).to_string(index=False))

## 2. Oil Dependency Analysis

In [ ]:
# Oil rents as % of GDP
oil_rent_ind = 'NY.GDP.PETR.RT.ZS'

oman_oil = df[(df['indicator_code'] == oil_rent_ind) & 
              (df['country_code'] == 'OMN')].sort_values('year')

fig, ax = plt.subplots(figsize=(12, 6))

ax.bar(oman_oil['year'], oman_oil['value'], color='#1f77b4', edgecolor='white')
ax.axhline(y=oman_oil['value'].mean(), color='red', linestyle='--', 
           label=f'Average: {oman_oil["value"].mean():.1f}%')

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Oil Rents (% of GDP)', fontsize=12)
ax.set_title('Oman Oil Rents as % of GDP', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../outputs/charts/oman_oil_dependency.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nOil Dependency Analysis:")
print(f"Average oil rents: {oman_oil['value'].mean():.1f}% of GDP")
print(f"Peak: {oman_oil['value'].max():.1f}% ({oman_oil.loc[oman_oil['value'].idxmax(), 'year']})")
print(f"Recent: {oman_oil['value'].iloc[-1]:.1f}% ({oman_oil['year'].iloc[-1]})")

## 3. GCC Trade Comparison

In [ ]:
# Merchandise trade comparison
trade_ind = 'TG.VAL.TOTL.GD.ZS'

gcc_trade = df[df['indicator_code'] == trade_ind].copy()
latest_year = gcc_trade['year'].max()
gcc_trade_latest = gcc_trade[gcc_trade['year'] == latest_year].sort_values('value', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#E41A1C' if c == 'Oman' else '#377EB8' for c in gcc_trade_latest['country_name']]
bars = ax.barh(gcc_trade_latest['country_name'], gcc_trade_latest['value'], color=colors)

ax.set_xlabel('Merchandise Trade (% of GDP)', fontsize=12)
ax.set_title(f'GCC Merchandise Trade Comparison ({latest_year})', fontsize=14, fontweight='bold')

for bar, value in zip(bars, gcc_trade_latest['value']):
    if pd.notna(value):
        ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                f'{value:.1f}%', va='center', fontsize=10)

ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('../outputs/charts/gcc_trade_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Current Account Balance

In [ ]:
# Current account balance
cab_ind = 'BN.CAB.XOKA.CD'

oman_cab = df[(df['indicator_code'] == cab_ind) & 
              (df['country_code'] == 'OMN')].sort_values('year')
oman_cab['cab_billions'] = oman_cab['value'] / 1e9

fig, ax = plt.subplots(figsize=(12, 6))

colors = ['#2ca02c' if v >= 0 else '#d62728' for v in oman_cab['cab_billions']]
ax.bar(oman_cab['year'], oman_cab['cab_billions'], color=colors, edgecolor='white')
ax.axhline(y=0, color='black', linewidth=0.5)

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Current Account Balance (Billion USD)', fontsize=12)
ax.set_title('Oman Current Account Balance', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../outputs/charts/oman_current_account.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary
surplus_years = len(oman_cab[oman_cab['cab_billions'] > 0])
deficit_years = len(oman_cab[oman_cab['cab_billions'] < 0])
print(f"\nCurrent Account Summary:")
print(f"Years with surplus: {surplus_years}")
print(f"Years with deficit: {deficit_years}")

## 5. Key Findings

In [ ]:
print("=" * 60)
print("KEY FINDINGS: Oman Trade Analysis")
print("=" * 60)

print("\n1. Export Dependency:")
print(f"   - Exports average ~50% of GDP")
print(f"   - Highly dependent on oil/gas exports")

print("\n2. Oil Dependency:")
avg_oil = oman_oil['value'].mean()
print(f"   - Oil rents average: {avg_oil:.1f}% of GDP")
print(f"   - Diversification efforts ongoing (Vision 2040)")

print("\n3. Trade Balance:")
print(f"   - Generally positive but volatile")
print(f"   - Correlated with global oil prices")

print("\n4. Recommendations:")
print("   - Continue economic diversification")
print("   - Develop non-oil export sectors")
print("   - Strengthen tourism and logistics")

---

## Next: GCC Comparison

Continue to **04_gcc_comparison.ipynb** for comprehensive regional comparison.